# Catching Rare Events
### BAI709 Data Modelling and Machine Learning

---

**The business question.** A risk team wants a model that catches the rare applicants who default when almost everyone repays. A model that says "everyone will repay" is right nearly every time and is no use at all. How do you build and judge a model for a rare event?

**Read this before you start: the data in this notebook is constructed.** Real home equity defaults in the loan file are common (about one in five). To practise on a *rare* event, this notebook builds a thinned copy of the file in which most of the defaulters have been deleted. The default rate in the thinned copy is not a fact about lending. It is never saved as a dataset, and real rare-event data (fraud, for example, at a few cases in a thousand) is harder still.

---

### How this notebook works

Every part follows five steps.

| Step | What it is |
|---|---|
| 1 | **Load and look** |
| 2 | **Run the model** |
| 3 | **Read the output in plain English** |
| 4 | **Check it**: against a calculation you can do by hand, and against the other notebooks |
| 5 | **Say it to a manager** |

- Run cells in order with **Shift+Enter**. The data is in `../data/`.
- **Copilot is on.** Where a cell says *Ask Copilot*, read the code it gives you and check the answer against a number you can verify.
- **Your turn** cells are for you to change and re-run. **Stretch** is blank for quick finishers.
- The costs in this notebook are **invented for teaching**.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, precision_recall_curve

import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.width", 140)
print("Libraries loaded.")

### The fixed settings (one cell, leave it alone)

- **Split:** 70% training, 30% test, `random_state=1`, stratified, done **before** any cleaning.
- **Logistic regression:** standardised inputs, `max_iter=5000`. **Forest:** 300 trees, `min_samples_leaf=5`.
- **Class weights:** off unless a cell says "balanced".
- **Cross-validation:** five folds on the training rows only, scored by AUC.
- **Costs (invented):** a missed default loses 10,000; turning away a good applicant loses 1,000.

In [ ]:
CV = StratifiedKFold(5, shuffle=True, random_state=1)
COST_MISSED, COST_FALSE_ALARM = 10_000, 1_000        # invented

def make_models(weights=None):
    # weights=None is the usual model; weights="balanced" asks the model to take rare cases more seriously
    return {
        "Logistic": make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000, class_weight=weights)),
        "Forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=1,
                                         n_jobs=-1, class_weight=weights),
    }

def prepare_hmeq(data, seed=1):
    # The same preparation as the other loan notebooks: split first, learn from training rows only
    y = data["BAD"]; X = data.drop(columns="BAD")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=seed, stratify=y)
    numeric = ["LOAN", "MORTDUE", "VALUE", "YOJ", "DEROG", "DELINQ", "CLAGE", "NINQ", "CLNO", "DEBTINC"]
    medians = X_train[numeric].median()
    parts = []
    for part in (X_train, X_test):
        part = part.copy()
        part["DEBTINC_missing"] = part["DEBTINC"].isna().astype(int)
        part[numeric] = part[numeric].fillna(medians)
        for col in ("REASON", "JOB"):
            part[col] = part[col].fillna("Missing")
        parts.append(part)
    X_train = pd.get_dummies(parts[0], columns=["REASON", "JOB"], dtype=int)
    X_test = pd.get_dummies(parts[1], columns=["REASON", "JOB"], dtype=int).reindex(columns=X_train.columns, fill_value=0)
    drop = ["REASON_DebtCon", "JOB_Other"]
    return X_train.drop(columns=drop), X_test.drop(columns=drop), y_train, y_test

def outcomes(y, p, cutoff):
    flagged = p >= cutoff
    return {"accuracy": float((flagged == y).mean()),
            "caught": int((flagged & (y == 1)).sum()),
            "false alarms": int((flagged & (y == 0)).sum())}
print("Settings and helper functions ready.")

---
# Exercise: a thinned loan file with 3% defaults

## Part 1: Build the rare-event file and look at it

| Step | What it is | What we do in this part |
|---|---|---|
| 1 | **Load and look** | Build the thinned file and count what is in it. |
| 2 | **Run the model** | Split it with the fixed settings and the same preparation. |
| 3 | **Read the output in plain English** | Read the baseline: what does 'approve everyone' score? |
| 4 | **Check against JMP** | Check the counts add up by hand. |
| 5 | **Say it to a manager** | Say why accuracy is a poor yardstick here. |

### Step 1: Load and look

We keep **every repaying applicant** and a **random sample of 148 defaulters**, then shuffle. This is a *constructed* file.

In [ ]:
full = pd.read_csv("../data/hmeq_full.csv")
defaulters = full[full["BAD"] == 1]
repayers = full[full["BAD"] == 0]

# CONSTRUCTED FILE: all the repayers plus 148 sampled defaulters, shuffled. Not a real default rate.
rare = pd.concat([repayers, defaulters.sample(148, random_state=1)]).sample(frac=1, random_state=1).reset_index(drop=True)

print(f"Real file: {len(full):,} applicants, default rate {full['BAD'].mean():.1%}")
print(f"Constructed file: {len(rare):,} applicants, default rate {rare['BAD'].mean():.2%}  (this is the rare event)")

### Step 2: Run the preparation

In [ ]:
X_train, X_test, y_train, y_test = prepare_hmeq(rare)
print(f"Training: {len(X_train):,} applicants, {y_train.sum()} defaults")
print(f"Test:     {len(X_test):,} applicants, {y_test.sum()} defaults")

### Step 3: Read the output in plain English

In [ ]:
baseline = 1 - y_test.mean()
print(f"A model that predicts 'repays' for every single applicant is right {baseline:.1%} of the time on the test applicants.")
print(f"It catches none of the {y_test.sum()} defaulters. Any real model has to be judged by what it catches, not by accuracy.")
print(f"Only {y_test.sum()} defaulters sit in the test set, so every number in this notebook that depends on them is noisy.")

### Step 4: Check it by hand

In [ ]:
# The two groups must add up to everyone in the test set
repayers_in_test = int((y_test == 0).sum())
print(f"{repayers_in_test:,} repayers + {y_test.sum()} defaulters = {repayers_in_test + int(y_test.sum()):,} = {len(y_test):,} test applicants")
print(f"Baseline accuracy by hand: {repayers_in_test:,} / {len(y_test):,} = {repayers_in_test / len(y_test):.3f}")

### Step 5: Say it to a manager

*One sentence:* why is "97% accurate" not good news for a risk team?

*Your sentence here*

---
## Part 2: Fit two models and meet the accuracy trap

| Step | What it is | What we do in this part |
|---|---|---|
| 1 | **Load and look** | Reuse the prepared rows from Part 1. |
| 2 | **Run the model** | Fit a logistic regression and a forest; score at the usual 0.5 cut-off. |
| 3 | **Read the output in plain English** | Read accuracy alongside defaults caught. |
| 4 | **Check against JMP** | Check against the 97% baseline and the AUC. |
| 5 | **Say it to a manager** | Say what the forest at 0.5 is really doing. |

### Step 2: Run the models

**What to look for:** accuracy, then **defaults caught** (out of the defaulters in the test set), then **false alarms** (good applicants flagged).

In [ ]:
models = make_models()
probs = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    probs[name] = model.predict_proba(X_test)[:, 1]

rows = []
for name, p in probs.items():
    o = outcomes(y_test, p, 0.5)
    rows.append({"Model": name, "Test AUC": roc_auc_score(y_test, p),
                 "Accuracy at 0.5": o["accuracy"], f"Caught (of {y_test.sum()})": o["caught"], "False alarms": o["false alarms"]})
part2 = pd.DataFrame(rows).set_index("Model")
part2.round(3)

### Step 3: Read the output in plain English

In [ ]:
f = part2.loc["Forest"]; l = part2.loc["Logistic"]
print(f"The forest at 0.5 is {f['Accuracy at 0.5']:.1%} accurate, and it catches {int(f.iloc[2])} of {y_test.sum()} defaulters.")
print(f"'Approve everyone' is {baseline:.1%} accurate. The forest has barely moved off the baseline.")
print(f"The logistic model at 0.5 catches {int(l.iloc[2])} of {y_test.sum()} with {int(l['False alarms'])} false alarms.")
print(f"Yet the test AUC is a healthy {f['Test AUC']:.3f} for the forest and {l['Test AUC']:.3f} for the logistic model. AUC looks good while the default behaviour is poor, because AUC measures ranking and not what happens at 0.5.")

### Step 4: Check it

The forest's probabilities for defaulters are real but small, and almost never reach 0.5. Look at them to see why.

In [ ]:
pf = pd.Series(probs["Forest"], index=y_test.index)
print(f"Forest probability, defaulters: median {pf[y_test == 1].median():.3f}, highest {pf[y_test == 1].max():.3f}")
print(f"Forest probability, repayers:   median {pf[y_test == 0].median():.3f}, highest {pf[y_test == 0].max():.3f}")
print(f"Defaulters do get higher probabilities than repayers, but few cross 0.5 when only {y_test.mean():.1%} of applicants default.")

### Step 5: Say it to a manager

*Two sentences:* what is the forest at a 0.5 cut-off doing, and why does its AUC not warn us?

*Your two sentences here*

---
## Part 3: Move the cut-off

| Step | What it is | What we do in this part |
|---|---|---|
| 1 | **Load and look** | Use the probabilities you already have. |
| 2 | **Run the model** | Try cut-offs from 0.5 down to 0.03 and count what each catches. |
| 3 | **Read the output in plain English** | Read the trade: more caught, more false alarms, lower accuracy. |
| 4 | **Check against JMP** | Check one row by hand from the confusion matrix. |
| 5 | **Say it to a manager** | Say which cut-off you would use and why. |

The model does not decide the cut-off. **The cost of the mistakes does.** A missed default loses 10,000 and a false alarm loses 1,000 (both invented), so the lender should flag anyone whose chance of default is above about 1 in 11, which is 0.091.

In [ ]:
cutoffs = [0.5, 0.3, 0.2, 0.1, 0.05, 0.03]
sweep = []
for name, p in probs.items():
    for c in cutoffs:
        o = outcomes(y_test, p, c)
        missed = int(y_test.sum()) - o["caught"]
        sweep.append({"Model": name, "Cut-off": c, "Accuracy": o["accuracy"], "Caught": o["caught"],
                      "False alarms": o["false alarms"],
                      "Cost (invented)": missed * COST_MISSED + o["false alarms"] * COST_FALSE_ALARM})
sweep = pd.DataFrame(sweep)
display(sweep.round(3).style.format({"Cost (invented)": "{:,.0f}", "False alarms": "{:,}"}))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
for name in probs:
    s = sweep[sweep["Model"] == name]
    ax.plot(s["False alarms"], s["Caught"], marker="o", label=name)
    for _, r in s.iterrows():
        ax.annotate(f"{r['Cut-off']}", (r["False alarms"], r["Caught"]), textcoords="offset points", xytext=(5, -10), fontsize=8)
ax.set_xlabel("False alarms (good applicants flagged)"); ax.set_ylabel(f"Defaulters caught (of {y_test.sum()})")
ax.set_title("Lowering the cut-off catches more and flags more"); ax.legend()
plt.show()

### Step 3: Read the output in plain English

In [ ]:
for name in probs:
    s = sweep[sweep["Model"] == name].set_index("Cut-off")
    cheapest = s["Cost (invented)"].idxmin()
    print(f"{name}: at 0.5 it catches {s.loc[0.5, 'Caught']} of {y_test.sum()}; at 0.1 it catches {s.loc[0.1, 'Caught']} but raises {s.loc[0.1, 'False alarms']:,} false alarms "
          f"and accuracy falls to {s.loc[0.1, 'Accuracy']:.1%}.")
    print(f"   Cheapest cut-off on this grid: {cheapest} (cost {s.loc[cheapest, 'Cost (invented)']:,.0f}, against {s.loc[0.5, 'Cost (invented)']:,.0f} at 0.5).")

### Step 4: Check it by hand

One row of the table, rebuilt from the confusion matrix.

In [ ]:
tn, fp, fn, tp = confusion_matrix(y_test, probs["Forest"] >= 0.1).ravel()
print(f"Forest at 0.1: TN {tn:,}, FP {fp}, FN {fn}, TP {tp}")
print(f"Caught = TP = {tp}; false alarms = FP = {fp}; accuracy = (TN + TP) / all = {(tn + tp) / len(y_test):.3f}")
print(f"Cost = {fn} x {COST_MISSED:,} + {fp} x {COST_FALSE_ALARM:,} = {fn * COST_MISSED + fp * COST_FALSE_ALARM:,}")

### Your turn

Change the cut-off and re-run. Find the lowest cut-off at which the forest still keeps accuracy above 90%.

In [ ]:
CUTOFF = 0.1            # <- change me
o = outcomes(y_test, probs["Forest"], CUTOFF)
print(f"Forest at {CUTOFF}: accuracy {o['accuracy']:.1%}, caught {o['caught']} of {y_test.sum()}, false alarms {o['false alarms']:,}")

### Step 5: Say it to a manager

*Two sentences:* which cut-off would you recommend, and who pays for the false alarms?

*Your two sentences here*

---
## Part 4: Class weights

| Step | What it is | What we do in this part |
|---|---|---|
| 1 | **Load and look** | Use the same training rows. |
| 2 | **Run the model** | Refit both models with `class_weight='balanced'`. |
| 3 | **Read the output in plain English** | Read what changes at each cut-off and what happens to the probabilities. |
| 4 | **Check against JMP** | Check the average predicted probability against the true default rate. |
| 5 | **Say it to a manager** | Say whether class weights are a fix or a shortcut. |

> **Concept: class weights.** `class_weight="balanced"` tells the model that one rare default counts for as much as many repayers. The model then pays far more attention to the rare cases, and it flags many more applicants. That does the same job as **moving the cut-off**, and it has a cost: the predicted probabilities are no longer real probabilities. A forest trained with balanced weights may give a 40% chance to applicants whose true chance of default is 3%. Use the weights if you only need a ranking or a flag, and do not read the numbers as risks.

In [ ]:
balanced = {}
for name, model in make_models("balanced").items():
    model.fit(X_train, y_train)
    balanced[f"{name}, balanced weights"] = model.predict_proba(X_test)[:, 1]

all_probs = {**probs, **balanced}
rows = []
for name, p in all_probs.items():
    row = {"Model": name, "Test AUC": roc_auc_score(y_test, p), "Average precision": average_precision_score(y_test, p)}
    for c in (0.5, 0.1, 0.05):
        o = outcomes(y_test, p, c)
        row[f"At {c}: accuracy / caught / false alarms"] = f"{o['accuracy']:.3f} / {o['caught']} / {o['false alarms']:,}"
    rows.append(row)
pd.DataFrame(rows).set_index("Model")

### Step 3: Read the output in plain English

In [ ]:
fb = all_probs["Forest, balanced weights"]
o5 = outcomes(y_test, fb, 0.5)
print(f"The balanced forest at 0.5 catches {o5['caught']} of {y_test.sum()} with {o5['false alarms']} false alarms and {o5['accuracy']:.1%} accuracy.")
print("Balancing does what a lower cut-off does: more caught, more false alarms, lower accuracy.")
print(f"Its AUC ({roc_auc_score(y_test, fb):.3f}) hardly changes, because the ranking is much the same.")

### Step 4: Check the probabilities

A probability should match how often the event happens. The true default rate in the test set is the yardstick.

In [ ]:
print(f"True default rate in the test set: {y_test.mean():.1%}")
for name, p in all_probs.items():
    print(f"{name:<32} average predicted probability {p.mean():.1%}")

### Step 5: Say it to a manager

*One sentence:* why should you not quote a balanced model's probability as a risk?

*Your sentence here*

---
## Part 5: Precision and recall for rare events

| Step | What it is | What we do in this part |
|---|---|---|
| 1 | **Load and look** | Use the probabilities from the previous parts. |
| 2 | **Run the model** | Draw the precision-recall curve and work out average precision. |
| 3 | **Read the output in plain English** | Read where each model sits and why AUC looks kinder. |
| 4 | **Check against JMP** | Check precision at one cut-off by hand. |
| 5 | **Say it to a manager** | Say which measure you would quote to a risk committee. |

> **Concept: precision, recall and average precision.** *Recall* is the share of real defaulters that you flagged. *Precision* is the share of flagged applicants that really default. Lowering the cut-off raises recall and lowers precision, and the **precision-recall curve** draws that trade. **Average precision** summarises the curve; a model that guesses scores about the default rate (3% here), and a perfect model scores 1. Unlike AUC, it does not get flattered by the huge pile of easy repayers, so it is the fairer score for a rare event.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for name in ["Logistic", "Forest"]:
    pr, rc, _ = precision_recall_curve(y_test, all_probs[name])
    ax.plot(rc, pr, label=f"{name} (average precision {average_precision_score(y_test, all_probs[name]):.3f})")
ax.axhline(y_test.mean(), color="k", linestyle="--", linewidth=1, label="Guessing")
ax.set_xlabel("Recall (defaulters caught)"); ax.set_ylabel("Precision (flagged who really default)")
ax.set_title("Precision-recall curves"); ax.legend()
plt.show()

### Step 3: Read the output in plain English

In [ ]:
for name in ["Logistic", "Forest"]:
    print(f"{name}: AUC {roc_auc_score(y_test, all_probs[name]):.3f}, average precision {average_precision_score(y_test, all_probs[name]):.3f} "
          f"(guessing would score {y_test.mean():.3f}).")
print("Average precision is much lower than AUC. That gap is the honest picture of how hard a rare event is.")

### Step 4: Check precision by hand

In [ ]:
o = outcomes(y_test, all_probs["Forest"], 0.1)
print(f"Forest at 0.1: flagged {o['caught'] + o['false alarms']} applicants, of whom {o['caught']} really default.")
print(f"Precision = {o['caught']} / {o['caught'] + o['false alarms']} = {o['caught'] / (o['caught'] + o['false alarms']):.3f}; "
      f"recall = {o['caught']} / {y_test.sum()} = {o['caught'] / y_test.sum():.3f}")

### A check that fails quietly

A colleague works out average precision for the forest with the cell below. It runs without an error. Does the number look right beside the curve above? Fix it.

In [ ]:
# A colleague's average precision for the forest
forest = make_models()["Forest"].fit(X_train, y_train)
colleague_ap = average_precision_score(y_test, forest.predict(X_test))
print(f"Forest average precision: {colleague_ap:.3f}")

### Step 5: Say it to a manager

*One sentence:* if you could quote only one number about this model to a risk committee, which would it be and why?

*Your sentence here*

---
## Part 6: One split is one roll of the dice

| Step | What it is | What we do in this part |
|---|---|---|
| 1 | **Load and look** | Repeat the whole preparation and test with different splits. |
| 2 | **Run the model** | Fit the forest ten times and record the test AUC each time. |
| 3 | **Read the output in plain English** | Read the spread. |
| 4 | **Check against JMP** | Compare with the cross-validated spread of the logistic model. |
| 5 | **Say it to a manager** | Say how much one test number can be trusted. |

With only a few defaulters in the test set, the test AUC depends heavily on which defaulters happen to land there. We rebuild the split ten different ways (changing only the split seed) and see how far the forest's test AUC moves. We use 100 trees to keep it quick.

In [ ]:
spread = []
for seed in range(10):
    Xa, Xb, ya, yb = prepare_hmeq(rare, seed=seed)
    f = RandomForestClassifier(n_estimators=100, min_samples_leaf=5, random_state=1, n_jobs=-1).fit(Xa, ya)
    spread.append({"split seed": seed, "defaulters in test": int(yb.sum()), "forest test AUC": roc_auc_score(yb, f.predict_proba(Xb)[:, 1])})
spread = pd.DataFrame(spread).set_index("split seed")
display(spread.round(3))

cv_logit = cross_val_score(make_models()["Logistic"], X_train, y_train, cv=CV, scoring="roc_auc")
print(f"Logistic regression, five-fold cross-validation on the training rows: {cv_logit.mean():.3f} (sd {cv_logit.std():.3f})")

### Step 3: Read the output in plain English

In [ ]:
lo, hi = spread["forest test AUC"].min(), spread["forest test AUC"].max()
print(f"Across ten different splits the forest's test AUC runs from {lo:.3f} to {hi:.3f}: a range of {hi - lo:.3f}.")
print(f"Each test set holds only about {spread['defaulters in test'].mean():.0f} defaulters, so a single test AUC is noisy.")
print("The answer is to repeat the testing (for example repeated cross-validation) and to report a range, not one number.")

### Step 4: Check it

Does the spread shrink when the test set holds more defaulters? Repeat the same ten splits on the **real** loan file, where about a fifth of applicants default.

In [ ]:
real_spread = []
for seed in range(10):
    Xa, Xb, ya, yb = prepare_hmeq(full, seed=seed)
    f = RandomForestClassifier(n_estimators=100, min_samples_leaf=5, random_state=1, n_jobs=-1).fit(Xa, ya)
    real_spread.append(roc_auc_score(yb, f.predict_proba(Xb)[:, 1]))
print(f"Real loan file ({int(full['BAD'].sum()):,} defaulters): forest test AUC ranges from {min(real_spread):.3f} to {max(real_spread):.3f}, a range of {max(real_spread) - min(real_spread):.3f}.")
print(f"Constructed rare file ({int(rare['BAD'].sum())} defaulters): range of {hi - lo:.3f}.")
print("More defaulters in the test set means a steadier test number.")

### Ask Copilot

> *Suggested prompt:* "Using scikit-learn's RepeatedStratifiedKFold with 5 folds and 3 repeats, score my logistic regression pipeline on X_train and y_train by roc_auc, and print the mean and standard deviation."

Run it and **check** that the mean is close to the five-fold figure above.

In [ ]:
# Paste and run Copilot's repeated cross-validation here

### Step 5: Say it to a manager (fixed cell)

Two sentences in business language: how well can this model catch the rare defaults, and how sure are you?

*Your two sentences here*

### Talk it through

1. A model is "97% accurate" and flags nobody. What does the risk team lose, and what would you launch, change or stop?
2. Who should review the flagged applicants, and what happens to the applicants who were flagged wrongly?

---
## Stretch

Blank on purpose. Use Copilot to try something of your own and check it against a number you trust. Ideas: draw the cost against the cut-off as a line and mark its lowest point; try `class_weight={0: 1, 1: 5}` on the forest and describe how it sits between the plain and the balanced forest; thin the file to 1% defaults and see what happens to average precision.

In [ ]:
# Stretch: your idea here

---
*BAI709 Data Modelling and Machine Learning, Regent's University London*